# Project 18 — vLLM Concepts (continuous batching vs naive, TTFT/TPOT, PagedAttention)

```
Synthetic request stream (prompt/decode lengths, bursty arrivals, seeded)
  x latency model calibrated from a real TinyGPT forward on this machine
  -> scheduler A (naive static batching: wait for B, run to max-len, pad waste)
  -> scheduler B (continuous batching: iteration-level, backfill freed slots, no padding)
  -> PagedAttention accounting: block tables vs contiguous reservation + prefix sharing
  -> metrics: TTFT, TPOT, makespan, throughput, KV fragmentation -> results/sched.csv + sched.png
```

Goal: understand vLLM's three wins separately — less waiting (scheduling), less padding
(batching granularity), less waste (paging) — with numbers, not slogans.

Learning objectives:
1. Define TTFT (queue + prefill) vs TPOT (steady decode) and say which each optimization moves.
2. Simulate static vs continuous batching and state when the gap is biggest (variable lengths).
3. Compute KV-cache bytes with GQA and block-table waste vs contiguous reservation.
4. Read a scheduler comparison and argue batch-width / block-size / arrival-rate tradeoffs.
   Key lesson: static batching pays max-len tax on every request; continuous + paging
   convert that tax into throughput, especially under skew + bursts.


In [ ]:
import os # need os to make results/ and read cpu count
import random # need random to seed stdlib RNG
import time # need time to calibrate the latency model on real forwards
import csv # need csv to write sched.csv + blocks csv
import math # need math for ceil() in block accounting + exp()
import urllib.request # need urllib to try downloading TinyShakespeare (offline fallback)
import numpy as np # need numpy for workload sampling + medians
import torch # need torch for the calibration model (cost anchor)
import matplotlib # need matplotlib for scheduler figures
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for TTFT/TPOT/blocks figure
SEED = 18 # fixed seed: same workload + same calibration every run
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
torch.manual_seed(SEED) # seed torch
torch.set_num_threads(max(1, os.cpu_count() // 2)) # cap threads: stable timing
os.makedirs('./results', exist_ok=True) # create results/ for CSVs and PNGs
DEV = 'cuda' if torch.cuda.is_available() else 'cpu' # GPU if present else CPU
print(f"device={DEV} torch={torch.__version__}") # env line for the README


In [ ]:
# ---- latency model: calibrate prefill + decode cost on a real tiny forward ----
D, CTX = 64, 64 # calibration width + length (matches P16/P17 proxies)
emb = torch.nn.Embedding(61, D).to(DEV) # toy embedding (cost anchor, not a model)
blk = torch.nn.TransformerEncoderLayer(D, 4, 4 * D, batch_first=True).to(DEV) # one transformer layer on device
def timed_fwd(b, t, iters=20): # median forward seconds for shape (b, t)
    x = torch.randint(0, 61, (b, t)).to(DEV) # random ids at that shape
    for _ in range(5): # warmup iters
        _ = blk(emb(x)) # throwaway
    if DEV == 'cuda': # honest GPU timing needs sync
        torch.cuda.synchronize() # drain before timing
    ts = [] # samples
    with torch.no_grad(): # no graph
        for _ in range(iters): # timed iters
            t0 = time.time() # start
            _ = blk(emb(x)) # one layer forward
            if DEV == 'cuda': # per-iter sync on GPU
                torch.cuda.synchronize() # wait
            ts.append(time.time() - t0) # sample
    return float(np.median(ts)) # median resists outliers
t_pref = timed_fwd(4, 64) / (4 * 64) # seconds per prompt-token (prefill is parallel over prompt)
t_dec = timed_fwd(4, 65) / 4 # seconds per decode step for a batch row (one token per request per step)
t_pref = max(t_pref, 1e-6) # floor: never zero/negative in sim math
t_dec = max(t_dec, 1e-6) # floor: same guard
print(f"calibrated: prefill={t_pref * 1e6:.1f} us/tok decode={t_dec * 1e3:.3f} ms/step-row") # cost card for the README
# ---- workload: N requests with prompt/out lengths + arrival times (seeded, skewed) ----
N, BMAX = 48, 4 # 48 requests, max 4 running (small enough to simulate by hand-check)
rng = np.random.RandomState(SEED) # seeded workload RNG
prompts = rng.randint(16, 65, size=N).tolist() # prompt lengths 16..64 (prefill work varies 4x)
outs = rng.randint(8, 65, size=N).tolist() # decode lengths 8..64 (skewed: static batching pays max-len tax here)
arrs = sorted(rng.uniform(0, 0.15, size=N).tolist()) # arrivals spread over 0.15 s (tight enough to queue; loose enough to differ)
print(f"workload: N={N} prompt_med={float(np.median(prompts)):.0f} out_med={float(np.median(outs)):.0f} out_max={max(outs)}") # skew card (max vs med drives the tax)


In [ ]:
# ---- scheduler A: naive static batching (wait for B, pad to max, run to max) ----
def sched_static(prompts, outs, arrs, B=4): # fixed batches in arrival order, no backfill
    recs = [] # per-request records (ttft, tpot, done)
    t = 0.0 # simulated clock (seconds)
    i = 0 # next-request pointer
    while i < len(prompts): # consume the stream batch by batch
        j = min(i + B, len(prompts)) # batch slice [i, j)
        b = j - i # actual width (last batch may be short)
        t = max(t, arrs[j - 1]) # wait until the LAST member arrives (head-of-line blocking)
        mx = max(outs[i:j]) # decode steps = longest member (everyone pays max-len)
        lp = max(prompts[i:j]) # prefill tokens = longest prompt (padding waste below)
        t_pre = t_pref * sum(prompts[i:j]) # prefill cost ~ total prompt tokens (parallel, one pass)
        t_dec_tot = t_dec * mx * b / max(b, 1) * 1.0 # decode cost: mx steps; per-step cost scales with live rows (~b here)
        t_dec_tot = t_dec * mx # simplified: whole-batch step cost per iter (documented approximation)
        for k in range(i, j): # each member shares the batch clock
            ttft = (t - arrs[k]) + t_pre # queue wait + prefill until first token
            tpot = t_dec_tot / max(outs[k], 1) # its own TPOT spreads batch decode over ITS tokens (short reqs look bad: the tax)
            recs.append({'ttft': ttft, 'tpot': tpot, 'pad_waste': (mx - outs[k]) + (lp - prompts[k])}) # waste = decode pad + prefill pad
        t = t + t_pre + t_dec_tot # advance clock past this batch
        i = j # next batch
    return recs, t # (per-request records, makespan)
# ---- scheduler B: continuous batching (iteration-level: prefill on arrival, backfill on finish) ----
def sched_continuous(prompts, outs, arrs, B=4): # event-driven sim at decode-step granularity; records carry request ids (honest TPOT)
    order = sorted(range(len(prompts)), key=lambda k: arrs[k]) # arrival order
    t = 0.0 # clock
    nxt = 0 # pointer
    running = [] # live: dicts with id
    recs = {} # id -> record
    pending = [] # prefill queue (ids)
    while nxt < len(order) or pending or running: # drain everything
        while nxt < len(order) and arrs[order[nxt]] <= t: # admit arrivals
            pending.append(order[nxt]) # queue
            nxt += 1 # advance
        while pending and len(running) < B: # backfill
            k = pending.pop(0) # FIFO
            t += t_pref * prompts[k] # prefill cost
            running.append({'id': k, 'rem': outs[k], 'ak': arrs[k], 't1': t}) # live entry
        if not running: # idle: jump clock
            t = max(t, arrs[order[nxt]]) # fast-forward
            continue # re-loop
        t += t_dec # one joint decode step
        for r in [x for x in running if x['rem'] - 1 <= 0]: # finishers this step
            r['rem'] = 0 # mark done
            dur = t - r['t1'] # its decode duration
            recs[r['id']] = {'ttft': r['t1'] - r['ak'], 'tpot': dur / max(outs[r['id']], 1), 'pad_waste': 0} # honest per-id TPOT, zero pad waste
            running.remove(r) # free slot
        for r in running: # tick the survivors
            r['rem'] -= 1 # one token
    ordered = [recs[k] for k in range(len(prompts))] # back in request order
    return ordered, t # (records, makespan)
recsA, spanA = sched_static(prompts, outs, arrs) # run naive policy
recsB, spanB = sched_continuous(prompts, outs, arrs) # run continuous policy
assert spanB <= spanA # continuous must not lose on makespan (same work, less waiting/padding)
print(f"makespan static={spanA:.3f}s continuous={spanB:.3f}s (x{spanA / max(spanB, 1e-9):.2f})") # headline win


In [ ]:
# ---- metrics: TTFT/TPOT medians + throughput + waste; PagedAttention accounting ----
def agg(recs, span): # aggregate per-request records into serving metrics
    tt = [r['ttft'] for r in recs] # TTFT samples
    tp = [r['tpot'] for r in recs] # TPOT samples
    tot = sum(outs) # total decode tokens served
    return {'med_TTFT_ms': round(float(np.median(tt)) * 1e3, 2), # median queue+prefill
            'med_TPOT_ms': round(float(np.median(tp)) * 1e3, 2), # median steady-decode
            'makespan_s': round(span, 3), # wall time to drain
            'tok_s': round(tot / max(span, 1e-9), 1), # decode throughput
            'mean_pad_waste': round(float(np.mean([r['pad_waste'] for r in recs])), 1)} # avg padding tax per request
mA, mB = agg(recsA, spanA), agg(recsB, spanB) # both policies aggregated
print('static:    ', mA) # naive line
print('continuous:', mB) # vLLM-style line
assert mB['mean_pad_waste'] == 0 # continuous has no padding by construction (assert the model, not the world)
assert mB['tok_s'] >= mA['tok_s'] # throughput must not regress
# -- PagedAttention: block tables vs contiguous reservation + prefix sharing (KV in fp16) --
BLK = 16 # KV block size in tokens (vLLM default 16)
Lmax = max(p + o for p, o in zip(prompts, outs)) # contiguous reservation must fit the worst case
paged_b = 0 # paged KV bytes accumulator (block-granular)
for p, o in zip(prompts, outs): # per request
    paged_b += math.ceil((p + o) / BLK) * (2 * 2 * D * BLK) # blocks * (K+V * layers=2 * d * BLK) in fp16 bytes
contig_b = N * (2 * 2 * D * Lmax) # contiguous bytes at same dtype (fp16 = 2 B)
S, G = 32, 4 # shared prefix: 32 tokens common to a group of 4 (system prompt / few-shot)
share_save = (G - 1) * math.ceil(S / BLK) * (2 * 2 * D * BLK) # blocks saved by sharing one copy across G (copy-on-write)
print(f"KV fp16: contiguous={contig_b / 1e3:.1f} kB paged={paged_b / 1e3:.1f} kB (x{contig_b / max(paged_b, 1):.2f}) share-saves-{G}x{S}={share_save / 1e3:.1f} kB") # memory card
assert paged_b <= contig_b # paging must not use MORE than worst-case reservation
brows = [{'scheme': 'contiguous-max', 'KB': round(contig_b / 1e3, 1)}, # reservation row
         {'scheme': 'paged-b16', 'KB': round(paged_b / 1e3, 1)}, # paging row
         {'scheme': 'paged+shared-prefix', 'KB': round((paged_b - share_save) / 1e3, 1)}] # sharing row
srows = [{'policy': 'static', **mA}, {'policy': 'continuous', **mB}] # scheduler comparison rows
with open('./results/sched.csv', 'w', newline='') as f: # save scheduler table
    w = csv.DictWriter(f, fieldnames=list(srows[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(srows) # two policy rows
print('saved ./results/sched.csv') # confirm artifact
with open('./results/blocks.csv', 'w', newline='') as f: # save memory table
    w = csv.DictWriter(f, fieldnames=list(brows[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(brows) # three scheme rows
print('saved ./results/blocks.csv') # confirm artifact


In [ ]:
# ---- what-if: arrival-rate sweep (burst pressure) + block-size sweep + figure ----
rates = [] # burst sweep rows: compress arrivals into shorter windows (more overlap -> longer queues)
for window in [0.15, 0.05, 0.005]: # wide -> narrow arrival window (same 48 requests)
    sc = window / 0.15 # rescale factor vs base window
    aw = sorted([a * sc for a in arrs]) # compressed arrivals
    _, sA = sched_static(prompts, outs, aw) # naive makespan under burst
    _, sB = sched_continuous(prompts, outs, aw) # continuous makespan under burst
    rates.append({'window_s': window, 'static_s': round(sA, 3), 'cont_s': round(sB, 3), # makespans
                  'speedup': round(sA / max(sB, 1e-9), 2)}) # continuous win grows under burst
    print(rates[-1]) # one line per window
assert all(r['speedup'] > 1.1 for r in rates) # continuous must win at every burst level (saturated regime: stable ~1.3x)
bsweep = [] # block-size sweep: small blocks fit tight, big blocks waste more
for blk_tok in [8, 16, 32, 64]: # candidate block sizes
    waste = sum(math.ceil((p + o) / blk_tok) * blk_tok - (p + o) for p, o in zip(prompts, outs)) # total internal-fragmentation tokens
    bsweep.append({'block': blk_tok, 'waste_tok': waste}) # one row per size
    print(bsweep[-1]) # one line per size
assert bsweep[0]['waste_tok'] <= bsweep[-1]['waste_tok'] # smaller blocks must not waste more
with open('./results/sweeps.csv', 'w', newline='') as f: # save burst sweep
    w = csv.DictWriter(f, fieldnames=list(rates[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(rates) # one row per window
print('saved ./results/sweeps.csv') # confirm artifact
fig, ax = plt.subplots(1, 3, figsize=(14, 4)) # three panels: latency, throughput-waste, paging
ax[0].bar(['static TTFT', 'cont TTFT', 'static TPOT', 'cont TPOT'], # four latency bars
          [mA['med_TTFT_ms'], mB['med_TTFT_ms'], mA['med_TPOT_ms'], mB['med_TPOT_ms']]) # medians in ms
ax[0].set_ylabel('ms') # y label
ax[0].set_title('P18: TTFT/TPOT medians') # panel title
ax[0].tick_params(axis='x', rotation=15) # tilt labels
ax[1].bar(['static tok/s', 'cont tok/s'], [mA['tok_s'], mB['tok_s']]) # throughput bars
ax[1].set_ylabel('tok/s') # y label
ax[1].set_title(f"P18: throughput (pad waste {mA['mean_pad_waste']} vs {mB['mean_pad_waste']})") # title carries waste contrast
ax[2].bar([b['scheme'] for b in brows], [b['KB'] for b in brows]) # KV memory bars
ax[2].set_ylabel('KV kB (fp16)') # y label
ax[2].set_title('P18: paged KV + sharing') # panel title
ax[2].tick_params(axis='x', rotation=15) # tilt labels
fig.tight_layout() # avoid overlap
fig.savefig('./results/sched.png', dpi=120) # save figure
print('saved ./results/sched.png') # confirm artifact


Cleanup / next steps:
- Artifacts: `results/sched.csv` (static vs continuous), `results/blocks.csv` (KV paging),
  `results/sweeps.csv` (burst + block sweeps), `results/sched.png`.
- Model limits (stated, not hidden): decode-step cost is joint-batch approximated, prefill is
  serial at small B, and the proxy is 1 layer — the DIRECTION (continuous + paging wins,
  skew/bursts keep the ~1.3x stable while TTFT halves) transfers to real vLLM; the exact
  ratios need a GPU rerun.
- Next: Session 7 — multimodality + RAG + agents (P19 VLM, P20 RAG, P21 agent runtime).
